# 강의 02 · 실습 2 — 구조화 출력 · (3) 변형

## 1. 문제상황

- 시설팀은 매주 건물을 돌며 점검 메모를 남깁니다.
- 메모에는 조치가 필요한 발견 사항과 이상 없는 항목이 섞여 있고, 발견 사항마다 위치·증상·급한 정도·기한이 문장 속에 들어 있습니다.
- 담당자는 메모를 읽고 급한 순서대로 작업 지시서를 손으로 다시 씁니다.
- 모델에게 요약을 시켜 봤지만 급한 정도를 「높음」「중간」「곧」처럼 제각각 적어 와서 순서를 매길 수 없습니다.

## 2. 문제와 목표

- **문제**: 급한 정도가 자유 텍스트라서 값이 호출마다 달라지고, 프로그램이 발견 사항을 급한 순서로 정렬할 수 없습니다.
- **목표**: 점검 보고서의 모양을 클래스로 선언하고, 급한 정도를 1·2·3의 정수로만 받도록 값 검증기를 달고, 돌아온 보고서를 파이썬 객체로 받아 급한 순서로 정렬한 작업 지시서를 만드는 처리 흐름을 만듭니다.
    - 점검 보고서의 모양: 점검 장소, 발견 사항 리스트(발견 사항마다 위치·증상·급한 정도·기한), 이상 없는 항목 리스트.
    - 급한 정도: 1 = 당장 조치, 2 = 이번 주 안에 조치, 3 = 다음 점검까지 두어도 됨.
    - 작업 지시서: 발견 사항을 급한 순서로 정렬해 「[급한 정도] 위치 — 증상 (기한: …)」 한 줄씩 적은 목록.
- **목표 달성 여부의 판정 기준**: 점검 메모를 넣었을 때 발견 사항 셋과 이상 없는 항목 둘이 선언한 클래스의 인스턴스로 돌아오고, 작업 지시서가 급한 순서(당장 조치 → 이번 주 → 다음 점검까지)로 출력되며, 범위 밖의 급한 정도를 넣었을 때 검증 오류가 나는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec02_ex02_s3_diagram.svg)

## 4. 단계별 요구사항

1. **출력 스키마를 선언합니다.**
    - 발견 사항 한 개를 나타내는 `Finding`(`location`·`issue`·`priority`·`due`)과 보고서 전체를 나타내는 `InspectionReport`(`site`·`findings`·`ok_items`)를 `BaseModel`을 상속한 클래스로 선언합니다.
    - `priority`는 급한 정도를 뜻하는 `int`이며 1 = 당장 조치, 2 = 이번 주 안에 조치, 3 = 다음 점검까지 두어도 됨입니다.
    - 기한은 없을 수 있으므로 `due`는 `str | None`으로 둡니다.
2. **값 검증기를 답니다.**
    - `Finding`의 `priority`에 검증기를 붙여, 값이 1·2·3 중 하나가 아니면 `ValueError`를 내어 검증이 실패하게 합니다.
3. **스키마를 지정해 모델을 호출합니다.**
    - 시스템 프롬프트에 추출 규칙(당장 조치하면 1, 이번 주 안이면 2, 다음 점검까지 두어도 되면 3, 기한이 없으면 `due`는 null, 메모에 없는 내용은 지어내지 않는다)을 적고, `response_format`에 `InspectionReport`를 지정해 모델을 한 번 호출하고, 응답 문자열의 타입과 내용을 출력합니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
4. **돌아온 문자열을 객체로 되돌리고 작업 지시서를 만듭니다.**
    - 응답 문자열을 `InspectionReport.model_validate_json`으로 파싱해 인스턴스로 만들고, 발견 사항을 `priority` 오름차순으로 정렬해 「[급한 정도] 위치 — 증상 (기한: …)」 형식의 줄을 만듭니다.
    - 기한이 없는 항목에는 기한 표기를 붙이지 않습니다.
    - 인스턴스의 타입과 발견 사항 수·이상 없는 항목 수를 먼저 출력합니다.
5. **검증 실패를 관찰합니다.**
    - 위치가 빠지고 급한 정도가 9이며 이상 없는 항목이 리스트가 아닌 딕셔너리를 `model_validate`에 넣어 `ValidationError`를 관찰합니다.
    - 급한 정도 4를 값 검증기가 없는 클래스와 있는 클래스에 각각 넣어, 모양 검사와 값 검증의 차이를 봅니다.

## 5. 코드 골격 — 구조화 출력(pydantic) 4단

파이댄틱(pydantic)으로 구조화 출력을 받는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 스키마 선언 | 받아 낼 출력의 모양을 클래스로 선언하고 값 검증기를 답니다 | `class InspectionReport(BaseModel)`, `@field_validator` | 1, 2 |
| ② 스키마를 건 호출 | 모델 호출에 스키마를 지정해 출력 형식을 강제합니다 | `completion(..., response_format=InspectionReport)` | 3 |
| ③ 객체 수신·파싱 | 돌아온 문자열을 클래스로 되돌려 파이썬 객체로 씁니다 | `InspectionReport.model_validate_json(...)` | 4 |
| ④ 검증 실패 관찰 | 일부러 어긋난 값을 넣어 어디서 어떻게 걸리는지 봅니다 | `ValidationError`, `model_validate(broken)` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import json
import os
import re
from datetime import date

from dotenv import load_dotenv, find_dotenv
from litellm import completion
from pydantic import BaseModel, ValidationError, field_validator

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

MODEL = "openai/gpt-5.6-luna"
# 같은 OPENAI 키로 호출되는 대체 모델(2026-09-05 확인): openai/gpt-4o-mini · openai/gpt-4.1-mini · openai/gpt-5-mini · openai/gpt-5.4-mini
print("준비를 마쳤습니다.")

# 주어진 자료 — 값과 이름을 그대로 씁니다.
MEMO = """주간 시설 점검 메모 (본관)
- 3층 복도 비상등 두 개가 켜지지 않는다. 오늘 안에 교체해야 한다.
- 지하 1층 주차장 배수구에 물이 고여 있다. 이번 주 금요일까지 청소한다.
- 옥상 출입문 잠금장치가 뻑뻑하다. 다음 점검 때 다시 본다.
- 1층 로비 소화기 압력은 정상이다.
- 엘리베이터 정기 점검표는 이상 없다."""


### 단계 ① — 스키마 선언 (요구사항 1, 2)

- 출력의 필드 이름과 타입을 클래스로 선언합니다. `priority: int`로 선언하면 「높음」 같은 문자열은 모양 검사에서 걸립니다.
- 타입이 맞아도 값이 범위 밖일 수 있습니다. `@field_validator`가 1·2·3 밖의 정수를 걸러 냅니다. 검증기 안에서 `ValueError`를 내면 `ValidationError`로 보고됩니다.

In [ ]:
# 여기에 단계 ①(Finding·InspectionReport 스키마와 priority 값 검증기 선언)을 작성합니다.

### 단계 ② — 스키마를 건 호출 (요구사항 3)

- 시스템 프롬프트에 급한 정도의 숫자 규칙을 적습니다. 숫자의 뜻을 적어 두지 않으면 모델은 숫자를 제멋대로 매깁니다.
- `response_format`에 스키마 클래스를 넘기면 모델은 그 모양에 맞는 JSON 문자열만 돌려줍니다.
- 이 셀은 모델을 한 번 호출합니다.


In [ ]:
# 여기에 단계 ②(추출 규칙, 점검 메모, 스키마를 지정한 모델 호출)를 작성합니다.

### 단계 ③ — 객체 수신·파싱 (요구사항 4)

- `model_validate_json`이 문자열을 `InspectionReport` 인스턴스로 되돌리면서 모양 검사와 값 검증을 함께 합니다.
- 인스턴스의 `findings`는 `Finding` 객체의 리스트이므로 `priority`로 바로 정렬할 수 있습니다. 자유 텍스트였다면 정렬할 수 없었습니다.

In [ ]:
# 여기에 단계 ③(문자열을 인스턴스로 파싱, 급한 순서의 작업 지시서 만들기)을 작성합니다.

### 단계 ④ — 검증 실패 관찰 (요구사항 5)

- 모델을 부르지 않습니다. 손으로 만든 어긋난 딕셔너리를 `model_validate`에 넣어 검증이 어디서 걸리는지 봅니다.
- 첫 번째 입력은 위치가 빠지고 급한 정도가 9이며 이상 없는 항목이 문자열입니다. 두 번째 입력은 타입은 맞지만 범위 밖인 급한 정도 4입니다.
- 값 검증기가 없는 클래스는 두 번째 입력을 통과시키고, 값 검증기가 있는 클래스는 걸러 냅니다.
- 두 클래스의 결과가 각각 한 번씩, 모두 두 번 찍혀야 합니다. 하나만 찍히면 값 검증기가 걸러 내지 못한 것입니다.

In [ ]:
# 여기에 단계 ④(어긋난 입력으로 검증 실패 관찰, 모양 검사와 값 검증 대비)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 네 가지를 확인합니다.

1. 단계 ②의 출력은 `str`이며, 첫 글자가 여는 중괄호인 JSON 문자열입니다.
2. 단계 ③의 타입 줄에 `InspectionReport`가 찍히고, 발견 사항 수가 3, 이상 없는 항목 수가 2입니다. 작업 지시서가 `[1]` 3층 복도 비상등, `[2]` 지하 1층 주차장 배수구, `[3]` 옥상 출입문 순서로 찍히고, 기한이 적힌 항목의 줄에 기한이 붙고 기한이 없는 항목의 줄에는 기한 표기가 붙지 않습니다.
3. 단계 ④의 첫 번째 `ValidationError`에 `findings.0.location`, `findings.0.priority`, `ok_items` 세 필드가 함께 찍힙니다.
4. 단계 ④에서 같은 급한 정도 4를 넣었을 때, 모양 검사만 하는 클래스는 인스턴스를 만들고, 값 검증기가 있는 클래스는 `priority` 필드에서 `ValidationError`를 냅니다.

네 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다.